# KJH data diagnostics
Adapted from origin/kjh_dashboard for the final 9-input submission data.


## Source files


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/okm_cleaned_2021.csv").is_file())
sys.path.insert(0, str(ROOT))
from scripts import build_eda_evidence as eda
RAW_PATH = ROOT / "data/okm_augumented_2021.csv"
CLEAN_PATH = ROOT / "data/okm_cleaned_2021.csv"
FIGURES = ROOT / "report/figures"
EVIDENCE = ROOT / "report/evidence"
raw, clean = eda.load_data()
cleaned_file = pd.read_csv(CLEAN_PATH, encoding="utf-8-sig")


## Column profile


In [ ]:
files = pd.DataFrame([{"dataset": label, "rows": len(frame), "columns": len(frame.columns),
    "missing_cells": int(frame.isna().sum().sum()), "bytes": int(frame.memory_usage(deep=True).sum())}
    for label, frame in (("raw", raw), ("cleaned", cleaned_file))])
display(files)


## Time continuity


In [ ]:
profile = pd.DataFrame([{"dataset": label, "column": name, "dtype": str(frame[name].dtype),
    "missing": int(frame[name].isna().sum()), "unique": int(frame[name].nunique())}
    for label, frame in (("raw", raw), ("cleaned", cleaned_file)) for name in frame.columns])
display(profile)


## Shared schema


In [ ]:
time_check = pd.Series({"start": clean.datetime.min(), "end": clean.datetime.max(),
    "duplicate_timestamps": int(clean.datetime.duplicated().sum()),
    "non_hourly_steps": int(clean.datetime.diff().dropna().ne(pd.Timedelta(hours=1)).sum()),
    "restored_hour_labels": int(clean[eda.RESTORED].sum())})
display(time_check)


## Power distribution


In [ ]:
shared = sorted(set(raw.columns) & set(clean.columns))
display(pd.DataFrame({"shared_columns": pd.Series(shared)}))
print("Both files contain the same 6,168 hourly observations; cleaned adds quality flags.")


## Report figures


In [ ]:
power = clean[eda.POWER]
display(power.describe().to_frame("power_kw"))
print("Power correlation:", {lag: round(power.corr(power.shift(lag)), 3) for lag in (1, 24, 168)})


## 4. 요약 결과 자동 저장 (Reports & Figures)
- `reports/column_summary.csv` 저장 (개인정보 및 원본 행 데이터 배제, 통계치만)
- `reports/figures/`에 결측 패턴 및 주요 분포 시각화 저장

In [ ]:
EVIDENCE.mkdir(parents=True, exist_ok=True)
profile.to_csv(EVIDENCE / "column_profile.csv", index=False, encoding="utf-8-sig")
summary = eda.build()
display(Image(filename=str(FIGURES / "08_eda_overview.png")))
display(Image(filename=str(FIGURES / "09_eda_quality.png")))
